In [ ]:
# Cell 1: Install dependencies
!pip install -q langchain langchain-anthropic langgraph
print('Packages installed!')

In [ ]:
# Cell 2: Set API key — paste your key when prompted, then press Enter
import os
from getpass import getpass
os.environ['ANTHROPIC_API_KEY'] = getpass('Paste your Anthropic API key: ')
print('Key set!')

In [ ]:
# Cell 3: Define state schema and model
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_anthropic import ChatAnthropic

# Shared state — every node reads from and writes to this
class RLMState(TypedDict):
    claim: str
    counterarguments: str
    refined_claim: str
    tensions: str
    synthesis: str

model = ChatAnthropic(model='claude-haiku-4-5-20251001')
print('State schema and model ready!')

In [ ]:
# Cell 4: Define nodes and build the reasoning graph
from langgraph.graph import StateGraph, END

# Node 1: Generate strongest counterarguments to the claim
def counterargument(state: RLMState) -> dict:
    prompt = f"""You are a rigorous philosophical critic.

Claim: {state['claim']}

Generate the 2-3 strongest counterarguments to this claim. Be precise and direct."""
    response = model.invoke([HumanMessage(content=prompt)])
    return {'counterarguments': response.content}

# Node 2: Refine the claim in light of the counterarguments
def refinement(state: RLMState) -> dict:
    prompt = f"""You are a careful reasoning analyst.

Original claim: {state['claim']}

Counterarguments raised:
{state['counterarguments']}

Refine the original claim to address the strongest objections while preserving its core insight."""
    response = model.invoke([HumanMessage(content=prompt)])
    return {'refined_claim': response.content}

# Node 3: Surface remaining tensions after refinement
def tension_analysis(state: RLMState) -> dict:
    prompt = f"""You are a dialectical analyst.

Original claim: {state['claim']}
Counterarguments: {state['counterarguments']}
Refined claim: {state['refined_claim']}

Identify the remaining tensions and unresolved questions that persist even after refinement."""
    response = model.invoke([HumanMessage(content=prompt)])
    return {'tensions': response.content}

# Node 4: Synthesize everything into a final coherent position
def synthesis(state: RLMState) -> dict:
    prompt = f"""You are a synthesis expert.

Original claim: {state['claim']}
Counterarguments: {state['counterarguments']}
Refined claim: {state['refined_claim']}
Remaining tensions: {state['tensions']}

Produce a final synthesis that integrates all of the above into a coherent, nuanced position."""
    response = model.invoke([HumanMessage(content=prompt)])
    return {'synthesis': response.content}

# Build the graph
graph = StateGraph(RLMState)
graph.add_node('counterargument', counterargument)
graph.add_node('refinement', refinement)
graph.add_node('tension_analysis', tension_analysis)
graph.add_node('synthesis', synthesis)

graph.set_entry_point('counterargument')
graph.add_edge('counterargument', 'refinement')
graph.add_edge('refinement', 'tension_analysis')
graph.add_edge('tension_analysis', 'synthesis')
graph.add_edge('synthesis', END)

agent = graph.compile()
print('RLM reasoning graph built!')
print()
print('Flow: counterargument -> refinement -> tension_analysis -> synthesis')

In [ ]:
# Cell 5: Run the agent on an RLM claim
result = agent.invoke({
    'claim': 'Persistent memory is necessary for continuity-bearing reasoning.',
    'counterarguments': '',
    'refined_claim': '',
    'tensions': '',
    'synthesis': ''
})

print('=== ORIGINAL CLAIM ===')
print(result['claim'])

print('\n=== COUNTERARGUMENTS ===')
print(result['counterarguments'])

print('\n=== REFINED CLAIM ===')
print(result['refined_claim'])

print('\n=== REMAINING TENSIONS ===')
print(result['tensions'])

print('\n=== FINAL SYNTHESIS ===')
print(result['synthesis'])